# VLA book renders
Streamlined from `p40.ipynb` + `p42.ipynb`. For one attention head of π0's PaliGemma backbone:

1. Query / key / value bar renders for a handful of token rows (transparent background)
2. Dot products and attention values for those rows
3. Magenta attention heatmaps over the image patches, frame 150
4. The same heatmaps for a series of frames through the episode

In [ ]:
import os
import colorsys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm

import torch
from openpi.training import config as _config
from openpi.policies import policy_config as _policy_config
from openpi.models import model as _model
from openpi.models_pytorch.pi0_pytorch import make_att_2d_masks

## Config

In [ ]:
EPISODE_IDX = 0
MAIN_FRAME  = 150          # frame for the bars, the printed numbers, and the first heatmaps
SERIES_STEP = 40           # heatmap series: frames 0, 40, 80, ...
SERIES_FRAMES = None       # or an explicit list (overrides SERIES_STEP)

LAYER, HEAD = 8, 6         # PaliGemma attention head we're looking at
QUERY_ROW   = 769          # the token doing the looking ('un' in "uncap the pen")

# Token rows to render as bars / print numbers for. None = transparent spacer row.
# Rows 0-255 base cam patches, 256-511 left wrist, 512-767 right wrist, 768+ language.
ROWS = [32, None, 373, 374, 375, 376, None, 769, 770, 771, 772]

# Bars (pixels)
BAR_WIDTH  = 2180
BAR_ROW_H  = 40
BAR_GAP    = 12            # transparent gap between rows in the stacked image
LANG_COLOR = (0, 135, 135) # bar color for language tokens

# Heatmaps
HEATMAP_SCALE  = 4         # output pixels per model-input pixel (224px image -> 1016px PNG)
PATCH_GAP      = 2         # transparent gap between patches, in model-input pixels
HEATMAP_ALPHA  = 0.7
HEATMAP_NORM   = "camera"  # "camera": each image scaled to its own max (as in p42)
                           # "frame":  one scale shared by the 3 cameras of a frame
HEATMAP_VMAX   = None      # set a number to pin the scale across cameras AND frames
HIDE_LETTERBOX = False     # True -> black padding patches (top/bottom 2 rows) become transparent

PROMPT = "uncap the pen"
checkpoint_dir = "/home/stephen/openpi/pi0_aloha_pen_uncap_pytorch"
save_dir = "/home/stephen/robots/vla_book_renders"
os.makedirs(save_dir + "/bars", exist_ok=True)
os.makedirs(save_dir + "/heatmaps", exist_ok=True)

# name used in filenames, key in the model's input dict
CAMS = [("base", "base_0_rgb"), ("left_wrist", "left_wrist_0_rgb"), ("right_wrist", "right_wrist_0_rgb")]
N_PATCH = 256              # 16x16 patches per camera
LANG_START = N_PATCH * len(CAMS)

## Load episode + model

In [ ]:
ds = load_dataset("physical-intelligence/aloha_pen_uncap_diverse", split="train")
episode = ds.select(np.where(np.array(ds["episode_index"]) == EPISODE_IDX)[0])
n_frames = episode.num_rows
series_frames = list(SERIES_FRAMES) if SERIES_FRAMES is not None else list(range(0, n_frames, SERIES_STEP))
print(f"{n_frames} frames; heatmap series: {series_frames}")

In [ ]:
policy = _policy_config.create_trained_policy(_config.get_config("pi0_aloha_pen_uncap"), checkpoint_dir)
device = torch.device(policy._pytorch_device)
pt_model = policy._model
pali = pt_model.paligemma_with_expert.paligemma.language_model
pali.config._attn_implementation = "eager"   # so the attention module returns its weights

## Run the prefix and capture one head
`run_prefix(frame)` pushes the 3 images + prompt through PaliGemma once and returns, for `LAYER`/`HEAD`:
queries, keys, values (one row per token) and the post-softmax attention pattern.

In [ ]:
def frame_to_obs(frame):
    to_chw = lambda img: np.array(img).transpose(2, 0, 1)   # PIL HWC -> CHW
    return {
        "images": {cam: to_chw(frame[f"observation.images.{cam}"])
                   for cam in ["cam_high", "cam_left_wrist", "cam_right_wrist"]},
        "state": np.array(frame["observation.state"]),
        "prompt": PROMPT,
    }

def to_batched_torch(x):
    if isinstance(x, dict):
        return {k: to_batched_torch(v) for k, v in x.items()}
    return torch.from_numpy(np.array(x)).to(device)[None, ...]

def run_prefix(frame, layer=LAYER, head=HEAD):
    inputs = policy._input_transform(frame_to_obs(frame))
    observation = _model.Observation.from_dict(to_batched_torch(inputs))
    images, img_masks, lang_tokens, lang_masks, state = pt_model._preprocess_observation(observation, train=False)

    cap = {}
    attn_module = pali.layers[layer].self_attn
    grab = lambda name: (lambda module, inp, out: cap.__setitem__(name, out))
    hooks = [attn_module.q_proj.register_forward_hook(grab("q")),
             attn_module.k_proj.register_forward_hook(grab("k")),
             attn_module.v_proj.register_forward_hook(grab("v")),
             attn_module.register_forward_hook(grab("out"))]
    try:
        with torch.no_grad():
            prefix_embs, pad_masks, att_masks = pt_model.embed_prefix(images, img_masks, lang_tokens, lang_masks)
            att_4d = pt_model._prepare_attention_masks_4d(make_att_2d_masks(pad_masks, att_masks))
            positions = torch.cumsum(pad_masks, dim=1) - 1
            pt_model.paligemma_with_expert.forward(
                attention_mask=att_4d, position_ids=positions, past_key_values=None,
                inputs_embeds=[prefix_embs, None], use_cache=True)
    finally:
        for h in hooks:
            h.remove()

    attn = cap["out"][1]
    assert attn is not None, "No attention weights returned - is _attn_implementation 'eager'?"
    to_np = lambda t: t.detach().float().cpu().numpy()
    d = cap["k"].shape[-1]                               # head dim (256); K and V are shared across heads
    return {
        "images": {key: np.asarray(inputs["image"][key]) for _, key in CAMS},   # 224x224x3 uint8, as the model sees them
        "q": to_np(cap["q"][0, :, head * d:(head + 1) * d]),                    # (816, 256)
        "k": to_np(cap["k"][0]),                                                # (816, 256)
        "v": to_np(cap["v"][0]),                                                # (816, 256)
        "attn": to_np(attn[0, head]),                                           # (816, 816), rows = queries
        "valid": to_np(pad_masks[0]).astype(bool),                              # False for padded language slots
        "lang_ids": to_np(lang_tokens[0]).astype(int),
    }

main = run_prefix(episode[MAIN_FRAME])
print({k: v.shape for k, v in main.items() if hasattr(v, "shape")})

In [ ]:
def token_piece(token_id):
    """Text of a prompt token, via openpi's own tokenizer (falls back to the id)."""
    try:
        for t in policy._input_transform.transforms:
            tok = getattr(t, "tokenizer", None)
            if tok is not None:
                return repr(tok._tokenizer.id_to_piece(int(token_id)).replace("\u2581", " "))
    except Exception:
        pass
    return f"id {token_id}"

def describe_row(row, res=main):
    if row >= LANG_START:
        return f"text {token_piece(res['lang_ids'][row - LANG_START])}"
    cam, patch = divmod(row, N_PATCH)
    return f"{CAMS[cam][0]} patch ({patch // 16},{patch % 16})"

for r in [QUERY_ROW] + [r for r in ROWS if r is not None]:
    print(f"{r:4d}  {describe_row(r)}")

## 1. Query / key / value bars
Each bar is one token's 256-d vector, colored by what the token is: image patches take a color sampled from the patch,
language tokens are teal. Saved as one stacked image per type plus one PNG per row, all with transparent backgrounds.

In [ ]:
def patch_bright_average(img, exponent=2.0):
    """One representative color per 14x14 patch, weighted toward the patch's most colorful pixels.
    Colorless patches (e.g. the black letterbox padding) come back NaN."""
    patches = img.astype(np.float32).reshape(16, 14, 16, 14, 3)
    chroma = patches.max(axis=-1, keepdims=True) - patches.min(axis=-1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        weights = chroma**exponent / (chroma**exponent).sum(axis=(1, 3), keepdims=True)
    return (patches * weights).sum(axis=(1, 3))   # (16, 16, 3)

def token_colors(images, exponent):
    """(816, 3) color in [0,1] for every token row."""
    cols = [patch_bright_average(images[key], exponent).reshape(N_PATCH, 3) for _, key in CAMS]
    cols.append(np.tile(np.array(LANG_COLOR, dtype=np.float32), (816 - LANG_START, 1)))
    return np.vstack(cols) / 255.

def boost_color(c, saturation_boost=1.0, min_saturation=0.0, min_value=0.0):
    h, s, v = colorsys.rgb_to_hsv(*c)
    s = max(min(1.0, s * saturation_boost), min_saturation)
    v = max(v, min_value)
    return np.array(colorsys.hsv_to_rgb(h, s, v), dtype=np.float32)

def render_bar(vec, color, width=BAR_WIDTH, height=BAR_ROW_H, brightness_range=0.85, **boost):
    """One vector -> (height, width, 4) uint8 RGBA. Low values go dark, high values go toward white."""
    c = boost_color(color, **boost)
    lo, hi = vec.min(), vec.max()
    norm = (vec - lo) / (hi - lo) if hi - lo > 1e-12 else np.full_like(vec, 0.5)
    x = np.interp(np.linspace(0, 1, width), np.linspace(0, 1, len(vec)), norm)[:, None]
    dark, bright = c * (1 - brightness_range), c + (1 - c) * brightness_range
    rgb = np.where(x <= 0.5, dark + 2 * x * (c - dark), c + 2 * (x - 0.5) * (bright - c))
    rgba = np.concatenate([np.clip(rgb, 0, 1), np.ones((width, 1))], axis=1)
    return np.broadcast_to((255 * rgba).round().astype(np.uint8), (height, width, 4)).copy()

def render_bars(vectors, colors, rows=ROWS, gap=BAR_GAP, **style):
    """Stack bars for `rows`. Spacers (None) and rows with no color (NaN) are left transparent."""
    bars = []
    for r in rows:
        if r is None or np.isnan(colors[r]).any():
            bars.append(np.zeros((BAR_ROW_H, BAR_WIDTH, 4), np.uint8))
        else:
            bars.append(render_bar(vectors[r], colors[r], **style))
    spacer = np.zeros((gap, BAR_WIDTH, 4), np.uint8)
    stacked = np.vstack([p for b in bars for p in (b, spacer)][:-1])
    return stacked, bars

# color_exponent: how strongly the patch color favors the patch's most saturated pixels
# (dialed back for keys/values, where the orange gets intense)
BAR_STYLES = {
    "queries": dict(vectors=main["q"], color_exponent=2.0, saturation_boost=2.0, min_saturation=0.4, min_value=0.3),
    "keys":    dict(vectors=main["k"], color_exponent=0.5, saturation_boost=2.0, min_saturation=0.3, min_value=0.3),
    "values":  dict(vectors=main["v"], color_exponent=0.5, saturation_boost=1.5, min_saturation=0.3, min_value=0.3),
}

fig, axes = plt.subplots(len(BAR_STYLES), 1, figsize=(12, 2.2 * len(BAR_STYLES)))
for ax, (name, style) in zip(axes, BAR_STYLES.items()):
    style = dict(style)
    colors = token_colors(main["images"], style.pop("color_exponent"))
    stacked, bars = render_bars(style.pop("vectors"), colors, **style)

    Image.fromarray(stacked).save(f"{save_dir}/bars/{name}_stacked.png")
    for i, (r, bar) in enumerate(zip(ROWS, bars)):
        if r is not None:
            Image.fromarray(bar).save(f"{save_dir}/bars/{name}_{i:02d}_row{r:03d}.png")

    ax.imshow(stacked, aspect="auto"); ax.set_title(name, loc="left"); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Dot products and attention values
For the query at `QUERY_ROW` against each key in `ROWS`:

- **q·k** - raw dot product of the captured query and key (what p40 printed)
- **q·k/√d** - scaled the way the attention layer scales it (d = 256)
- **no-RoPE attn** - softmax of that column over all valid tokens. This is what the attention *would* be from these vectors alone.
- **attention** - what the model actually computed. It differs from the previous column because the model rotates
  queries and keys by their positions (RoPE) before taking the dot product; the captured vectors are pre-rotation.

In [ ]:
q, K = main["q"][QUERY_ROW], main["k"]
dots = K @ q                                   # (816,)
scaled = dots / np.sqrt(K.shape[1])
no_rope = np.where(main["valid"], np.exp(scaled - scaled[main["valid"]].max()), 0.0)
no_rope /= no_rope.sum()
attn_row = main["attn"][QUERY_ROW]

print(f"Query: row {QUERY_ROW} ({describe_row(QUERY_ROW)}) | layer {LAYER}, head {HEAD}, frame {MAIN_FRAME}\n")
print(f"{'row':>4}  {'key token':<26}{'q·k':>9}{'q·k/√d':>9}{'no-RoPE attn':>14}{'attention':>11}")
for r in [r for r in ROWS if r is not None]:
    print(f"{r:>4}  {describe_row(r):<26}{dots[r]:>9.2f}{scaled[r]:>9.2f}{no_rope[r]:>14.4f}{attn_row[r]:>11.4f}")

print(f"\nWhere the query's attention goes (sums to {attn_row.sum():.3f}):")
for i, (name, _) in enumerate(CAMS):
    seg = attn_row[i * N_PATCH:(i + 1) * N_PATCH]
    print(f"  {name:<12} total {seg.sum():.3f}   max patch {seg.max():.4f} (row {i * N_PATCH + seg.argmax()})")
print(f"  {'language':<12} total {attn_row[LANG_START:].sum():.3f}")

## 3. Attention heatmaps, frame 150
The query's attention to each image patch, drawn in magenta over the patch. One PNG per camera;
the gaps between patches are transparent.

In [ ]:
magenta_cmap = LinearSegmentedColormap.from_list("magenta_hot", [(1, 1, 1, 0), (0.5, 0, 0.5, 0.8), (1, 0, 1, 1)])

def heatmap_image(img, attn16, vmax):
    """img: (224,224,3) uint8. attn16: (16,16) attention. Returns RGBA uint8 with transparent gaps."""
    S = HEATMAP_SCALE
    p, step = 14 * S, (14 + PATCH_GAP) * S
    big = np.asarray(Image.fromarray(img).resize((224 * S, 224 * S), Image.BICUBIC), dtype=np.float32) / 255.

    overlay = magenta_cmap(np.clip(attn16 / vmax, 0, 1))                       # (16,16,4), one value per patch
    overlay = overlay.repeat(p, axis=0).repeat(p, axis=1)
    a = overlay[..., 3:] * HEATMAP_ALPHA
    blended = (255 * (big * (1 - a) + overlay[..., :3] * a)).round().astype(np.uint8)

    letterbox = img.reshape(16, 14, 16, 14, 3).max(axis=(1, 3, 4)) == 0          # all-black padding patches
    size = 16 * p + 15 * PATCH_GAP * S
    out = np.zeros((size, size, 4), np.uint8)
    for i in range(16):
        for j in range(16):
            if HIDE_LETTERBOX and letterbox[i, j]:
                continue
            out[i * step:i * step + p, j * step:j * step + p, :3] = blended[i * p:(i + 1) * p, j * p:(j + 1) * p]
            out[i * step:i * step + p, j * step:j * step + p, 3] = 255
    return out

def render_heatmaps(res, frame_idx):
    """Save one heatmap PNG per camera for this frame; returns the 3 RGBA arrays."""
    row = res["attn"][QUERY_ROW]
    outs = []
    for i, (name, key) in enumerate(CAMS):
        cam_attn = row[i * N_PATCH:(i + 1) * N_PATCH]
        vmax = HEATMAP_VMAX or (cam_attn.max() if HEATMAP_NORM == "camera" else row[:LANG_START].max())
        out = heatmap_image(res["images"][key], cam_attn.reshape(16, 16), vmax)
        Image.fromarray(out).save(f"{save_dir}/heatmaps/frame{frame_idx:03d}_{name}.png")
        outs.append(out)
    return outs

def show_heatmaps(rows_of_outs, labels):
    fig, axes = plt.subplots(len(rows_of_outs), 3, squeeze=False, figsize=(12, 4 * len(rows_of_outs)))
    for r, (outs, label) in enumerate(zip(rows_of_outs, labels)):
        for c, out in enumerate(outs):
            axes[r, c].imshow(out); axes[r, c].axis("off")
            axes[r, c].set_title(f"frame {label} - {CAMS[c][0]}", fontsize=10)
    plt.tight_layout(); plt.show()

show_heatmaps([render_heatmaps(main, MAIN_FRAME)], [MAIN_FRAME])

## 4. Heatmap series through the episode

In [ ]:
series_outs = []
print(f"{'frame':>5}  max attention per camera ({', '.join(n for n, _ in CAMS)})")
for fi in tqdm(series_frames):
    res = run_prefix(episode[fi])
    series_outs.append(render_heatmaps(res, fi))
    row = res["attn"][QUERY_ROW]
    tqdm.write(f"{fi:>5}  " + "  ".join(f"{row[i * N_PATCH:(i + 1) * N_PATCH].max():.4f}" for i in range(len(CAMS))))

show_heatmaps(series_outs, series_frames)